# Industrial Machine Failure Prediction
AI4I 2020 Predictive Maintenance Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df=pd.read_csv('../data/ai4i2020.csv')
df.shape

In [ ]:
df.head()

In [ ]:
print(df.info())
print('Missing:',df.isna().sum().sum())
print('Duplicates:',df.duplicated().sum())

In [ ]:
df['Machine failure'].value_counts()

In [ ]:
df.describe(include='all').T

## Feature selection
UDI and Product ID are identifiers. TWF/HDF/PWF/OSF/RNF are failure-mode indicators and are excluded from the main pre-failure model to reduce leakage risk.

In [ ]:
features=['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
target='Machine failure'
X=df[features]
y=df[target]

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
pre=ColumnTransformer([
    ("cat",OneHotEncoder(handle_unknown="ignore",drop="first"),["Type"]),
    ("num",StandardScaler(),[c for c in features if c!="Type"])
])
models={
"Logistic Regression":LogisticRegression(max_iter=2000,class_weight="balanced",random_state=42),
"Decision Tree":DecisionTreeClassifier(max_depth=6,class_weight="balanced",random_state=42),
"Random Forest":RandomForestClassifier(n_estimators=300,max_depth=10,min_samples_leaf=2,class_weight="balanced",random_state=42,n_jobs=-1)
}
for name,m in models.items():
    pipe=Pipeline([("pre",pre),("model",m)])
    pipe.fit(X_train,y_train)
    p=pipe.predict(X_test); prob=pipe.predict_proba(X_test)[:,1]
    print("\n",name)
    print(classification_report(y_test,p,digits=3))
    print("ROC-AUC:",round(roc_auc_score(y_test,prob),3))


## Result
Random Forest was selected as the final model because it achieved the highest F1-score (0.676) and ROC-AUC (0.965) among the tested models on the fixed stratified test split.